# 🖥️ Instructor Prep (local/any-GPU edition): Download & Quantize FLUX.2 [klein]-4B

**Run this once, on hardware with more headroom than a free Colab T4** (e.g. a lab workstation, a
rented A100/A6000/4090 box, whatever you have access to) **— not by participants.**

This downloads **FLUX.2 [klein]-4B** and quantizes its transformer and text encoder to 4-bit NF4,
saving everything to a **local folder**. The last section walks through getting that folder into
Google Drive afterward so `00_complete_workshop.ipynb` can find it there.

**Why FLUX.2 [klein]-4B instead of FLUX.1-schnell + T5-XXL + CLIP-L:** klein uses a **single** text
encoder (Qwen3-4B) instead of a T5-XXL / CLIP-L pair — one embedding tensor to generate, manipulate,
and feed into the pipeline, not two. It's also fully open (**Apache 2.0, not gated** — no license
click-through, no Hugging Face token required to download it) and its quantized footprint is small
enough (~5GB total: ~2.2GB text encoder + ~2.2GB transformer + ~0.16GB VAE) that a free Colab T4
should be able to hold the whole pipeline in memory at once, instead of the
T5 → unload → CLIP → unload → FLUX dance the FLUX.1 edition needed.

**Why bother with different hardware at all:** quantizing a model needs a large *transient* memory
buffer beyond its own final size while the bf16 weights are loaded and converted — the text encoder
and transformer here are each roughly 7.5GB to download in bf16. Any GPU with real headroom (24GB+)
sidesteps that transient-memory problem outright, so this notebook is the "yes, but just give it more
room" path rather than fighting a small GPU's ceiling.

**Requirements:** Python with a CUDA-capable PyTorch already installed and working
(`torch.cuda.is_available()` should be `True` — this notebook doesn't install/reinstall PyTorch
itself, since getting the CUDA build right for *your* machine is environment-specific and easy to
break by overwriting). Everything else it needs, it installs itself in the next cell.


In [ ]:
# @title 1. Check the GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}")
    print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")
else:
    raise RuntimeError(
        "No CUDA GPU visible to PyTorch. This notebook needs a working CUDA + PyTorch install — "
        "fix that first (this notebook won't install/reinstall PyTorch itself)."
    )


In [ ]:
# @title 2. Install packages (pinned to versions verified compatible with each other)
# NOT installing torch/torchvision here — see the requirements note above.
#
# diffusers>=0.40.0 is required for Flux2KleinPipeline / Flux2Transformer2DModel / AutoencoderKLFlux2.
# That version requires huggingface-hub>=1.23.0,<2.0 — and transformers releases in the 4.x line all
# cap huggingface-hub at <1.0, which makes 4.x genuinely impossible to install alongside diffusers
# 0.40+ (pip can end up with an inconsistent combination instead of a clean resolver error if an
# older huggingface-hub is already present from a previous install in this environment, which is what
# produces a confusing `ImportError: cannot import name 'get_cached_repo_tree'` deep inside diffusers
# rather than an upfront pip conflict). So this pins transformers into the 5.x line on purpose — not
# to avoid it. The one thing that change costs us: 5.x's rewritten weight converter doesn't implement
# the save-direction path for bitsandbytes-quantized models yet, surfacing as a bare
# `NotImplementedError` from `ConversionOps.reverse_op` if you save one the normal way. The transformer
# + assemble cell below works around that directly (falls back to `save_original_format=False`), so
# it isn't a reason to avoid 5.x here.
#
# No -U: it lets pip decide it also needs to upgrade unrelated packages (torch included) to
# satisfy some dependency's minimum version -- on a machine with a specific CUDA-matched torch
# build already installed, that can silently replace it with an incompatible one (surfaces as
# a RuntimeError about overriding a dispatch key the moment you `import torch` next). The
# exact/ranged pins below are already enough to force a real resolve of diffusers/transformers/
# huggingface_hub against each other without touching torch.
%pip install -q "diffusers==0.40.0" "transformers==5.16.1" "huggingface_hub>=1.23.0,<2.0" "accelerate==1.12.0" bitsandbytes peft

import transformers, diffusers, huggingface_hub
print(f"transformers {transformers.__version__}, diffusers {diffusers.__version__}, huggingface_hub {huggingface_hub.__version__}")
assert transformers.__version__ == "5.16.1", (
    f"Expected transformers==5.16.1 but got {transformers.__version__} — some other install in this "
    "environment pulled in a different version, which given the huggingface-hub cross-dependency "
    "above (see comment) is worth resolving before continuing. "
    'Run `%pip install -q -U "transformers==5.16.1"` and restart the kernel.'
)


In [ ]:
# @title 3. Imports, and target path
import os, shutil, gc
from pathlib import Path

# Where this saves to — change this to wherever you want the files to land locally.
MODELS_DIR = Path.home() / "latent_vandalism_models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# One repo id, one saved folder — klein-4B's text encoder isn't shared with anything else the way
# T5-XXL used to be shared across FLUX.1 and SD3.5, so there's no reason to split it into its own
# reusable folder any more.
FLUX_KLEIN_REPO = "black-forest-labs/FLUX.2-klein-4B"
FLUX_KLEIN_MODEL_PATH = MODELS_DIR / "FLUX.2-klein-4B-nf4"

device = "cuda"
print(f"Target folder: {FLUX_KLEIN_MODEL_PATH}")


In [ ]:
# @title 4. Hugging Face login (optional — this repo isn't gated)
# Unlike black-forest-labs/FLUX.1-schnell, black-forest-labs/FLUX.2-klein-4B is Apache 2.0 and fully
# open — no license click-through, no token required to download it. Logging in still helps avoid
# anonymous rate limits on a ~15GB download, so it's offered here, but feel free to skip this cell.
from huggingface_hub import login

try:
    from huggingface_hub import HfFolder
    hf_token = HfFolder.get_token()
except Exception:
    hf_token = None

if not hf_token:
    from getpass import getpass
    hf_token = getpass('Optional: paste a Hugging Face token (hf_...), or leave blank and press Enter to skip: ')

if hf_token:
    login(token=hf_token)
    print("✓ Logged in to Hugging Face")
else:
    print("Skipping login — downloading anonymously.")


## Qwen3-4B text encoder (klein's only text encoder, 4-bit NF4)

klein uses one decoder-only language model as its text encoder instead of FLUX.1's T5-XXL + CLIP-L
pair. ~7.5GB to download in bf16; quantized down to roughly 2.2GB.


In [ ]:
from transformers import Qwen3ForCausalLM, Qwen2TokenizerFast, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

if (FLUX_KLEIN_MODEL_PATH / "text_encoder").exists():
    print(f"✓ {FLUX_KLEIN_MODEL_PATH / 'text_encoder'} already exists — loading it (needed below to assemble the full pipeline)")
    # Joined path passed directly (no subfolder= kwarg): diffusers/transformers' subfolder-joining
    # for a *local* directory has been unreliable for some component classes as of this writing —
    # passing the already-joined path sidesteps it entirely, for local loads everywhere in this
    # notebook.
    tokenizer = Qwen2TokenizerFast.from_pretrained(FLUX_KLEIN_MODEL_PATH / "tokenizer", local_files_only=True)
    text_encoder = Qwen3ForCausalLM.from_pretrained(
        FLUX_KLEIN_MODEL_PATH / "text_encoder", local_files_only=True, device_map={"": 0},
    )
else:
    print("Downloading Qwen3-4B (bf16, ~7.5GB) + quantizing to NF4...")
    tokenizer = Qwen2TokenizerFast.from_pretrained(FLUX_KLEIN_REPO, subfolder="tokenizer")
    text_encoder = Qwen3ForCausalLM.from_pretrained(
        FLUX_KLEIN_REPO, subfolder="text_encoder",
        quantization_config=bnb_config, device_map={"": 0},
    )
    # Assigned rather than a bare `text_encoder.eval()` statement: Jupyter/Colab caches the result of
    # any cell ending in a bare expression (into Out[]/_/__/___), which would keep this model pinned
    # in GPU memory even after a later `del` — the same issue the FLUX.1 prep notebook hit.
    text_encoder = text_encoder.eval()
    tokenizer.save_pretrained(FLUX_KLEIN_MODEL_PATH / "tokenizer")
    text_encoder.save_pretrained(FLUX_KLEIN_MODEL_PATH / "text_encoder")
    print(f"✓ Saved to {FLUX_KLEIN_MODEL_PATH}")


In [ ]:
# @title Free the raw download from local disk (the quantized copy above is what actually matters)
hf_cache = Path.home() / ".cache" / "huggingface"
if hf_cache.exists():
    freed = sum(f.stat().st_size for f in hf_cache.rglob('*') if f.is_file()) / 1024**3
    shutil.rmtree(hf_cache, ignore_errors=True)
    print(f"✓ Cleared {hf_cache} ({freed:.1f} GB freed)")


## VAE + scheduler (small, kept as-is — no quantization needed)


In [ ]:
from diffusers import AutoencoderKLFlux2, FlowMatchEulerDiscreteScheduler

if (FLUX_KLEIN_MODEL_PATH / "vae").exists():
    print(f"✓ {FLUX_KLEIN_MODEL_PATH / 'vae'} already exists — loading it (needed below to assemble the full pipeline)")
    vae = AutoencoderKLFlux2.from_pretrained(FLUX_KLEIN_MODEL_PATH / "vae", torch_dtype=torch.float16, local_files_only=True)
else:
    print("Downloading the VAE (small, ~160MB)...")
    vae = AutoencoderKLFlux2.from_pretrained(FLUX_KLEIN_REPO, subfolder="vae", torch_dtype=torch.float16)
    vae.save_pretrained(FLUX_KLEIN_MODEL_PATH / "vae")

if (FLUX_KLEIN_MODEL_PATH / "scheduler").exists():
    scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(FLUX_KLEIN_MODEL_PATH / "scheduler", local_files_only=True)
else:
    scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(FLUX_KLEIN_REPO, subfolder="scheduler")
    scheduler.save_pretrained(FLUX_KLEIN_MODEL_PATH / "scheduler")


## FLUX.2 [klein]-4B transformer (4-bit NF4) — then assemble + save the full pipeline

Unlike the FLUX.1 edition, there's no need to free and reload the text encoder around this step: the
whole quantized pipeline (text encoder + transformer + VAE) comes out to roughly 5GB, comfortably
resident all at once even during quantization.


In [ ]:
from diffusers import Flux2KleinPipeline, Flux2Transformer2DModel, AutoencoderKLFlux2, FlowMatchEulerDiscreteScheduler
from diffusers import BitsAndBytesConfig as DiffusersBnbConfig
from transformers import Qwen3ForCausalLM, Qwen2TokenizerFast

# Defensive reload: Jupyter runs cells in whatever order you *execute* them, not necessarily their
# order on the page — after a runtime restart, or if this cell gets run before the "Qwen3-4B text
# encoder" / "VAE + scheduler" cells above (easy to do by accident), `text_encoder`/`tokenizer`/
# `vae`/`scheduler` won't exist yet and this would otherwise fail with a bare NameError. Pick them
# back up from disk instead, as long as an earlier run already saved them there.
def _require(name, subfolder, loader, **kwargs):
    if name in globals():
        return globals()[name]
    path = FLUX_KLEIN_MODEL_PATH / subfolder
    if not path.exists():
        raise RuntimeError(
            f"`{name}` isn't defined yet and {path} doesn't exist on disk either — run the cell "
            f"above that produces `{name}` (look for the '{subfolder}' section) before this one."
        )
    print(f"(`{name}` wasn't in memory — reloading it from {path})")
    # Joined path, no subfolder= kwarg — see the note in the "Qwen3-4B text encoder" cell above.
    return loader(path, local_files_only=True, **kwargs)

tokenizer = _require("tokenizer", "tokenizer", Qwen2TokenizerFast.from_pretrained)
text_encoder = _require("text_encoder", "text_encoder", Qwen3ForCausalLM.from_pretrained, device_map={"": 0})
vae = _require("vae", "vae", AutoencoderKLFlux2.from_pretrained, torch_dtype=torch.float16)
scheduler = _require("scheduler", "scheduler", FlowMatchEulerDiscreteScheduler.from_pretrained)

if (FLUX_KLEIN_MODEL_PATH / "transformer").exists():
    print(f"✓ {FLUX_KLEIN_MODEL_PATH / 'transformer'} already exists — loading it to assemble the full pipeline")
    transformer = Flux2Transformer2DModel.from_pretrained(
        FLUX_KLEIN_MODEL_PATH / "transformer", torch_dtype=torch.float16, local_files_only=True,
    )
else:
    print("Downloading the FLUX.2 [klein]-4B transformer (bf16, ~7.2GB) + quantizing to NF4...")
    transformer_bnb_config = DiffusersBnbConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
    )
    transformer = Flux2Transformer2DModel.from_pretrained(
        FLUX_KLEIN_REPO, subfolder="transformer",
        quantization_config=transformer_bnb_config, torch_dtype=torch.float16,
    )

# is_distilled=True: klein is a distilled model (same idea as FLUX.1-schnell) — this tells the
# pipeline to skip real classifier-free guidance at generation time, matching how the source repo's
# own model_index.json describes it. Passing all five components in directly (rather than pointing
# at a folder) means save_pretrained below writes a consistent model_index.json even though
# text_encoder/tokenizer/vae/scheduler were already saved separately above.
flux_pipe = Flux2KleinPipeline(
    scheduler=scheduler, vae=vae,
    text_encoder=text_encoder, tokenizer=tokenizer,
    transformer=transformer,
    is_distilled=True,
)

# Save each component ourselves rather than flux_pipe.save_pretrained(): that method re-saves every
# component with save_original_format=True by default, which raises NotImplementedError for a
# bitsandbytes-quantized transformers model on some transformers versions — its newer weight-
# conversion system doesn't implement reversing a quantized state dict back to the "original"
# checkpoint format yet (surfaces as ConversionOps.reverse_op raising NotImplementedError, several
# frames down inside revert_weight_conversion). We only need flux_pipe for its pipeline-level
# metadata (model_index.json) at the end — the components are saved explicitly here instead.
tokenizer.save_pretrained(FLUX_KLEIN_MODEL_PATH / "tokenizer")
vae.save_pretrained(FLUX_KLEIN_MODEL_PATH / "vae")
scheduler.save_pretrained(FLUX_KLEIN_MODEL_PATH / "scheduler")
transformer.save_pretrained(FLUX_KLEIN_MODEL_PATH / "transformer")
try:
    text_encoder.save_pretrained(FLUX_KLEIN_MODEL_PATH / "text_encoder")
except NotImplementedError:
    print("  transformers couldn't reverse the quantized state dict to its original checkpoint format")
    print("  on this version — retrying with save_original_format=False, which skips that step (safe")
    print("  here: we only ever reload this with this same transformers install, not round-tripping")
    print("  through the Hub's canonical format).")
    text_encoder.save_pretrained(FLUX_KLEIN_MODEL_PATH / "text_encoder", save_original_format=False)
flux_pipe.save_config(FLUX_KLEIN_MODEL_PATH)

print(f"✓ Saved full pipeline to {FLUX_KLEIN_MODEL_PATH}")


In [ ]:
# @title Free the raw download from local disk (the quantized copy above is what actually matters)
hf_cache = Path.home() / ".cache" / "huggingface"
if hf_cache.exists():
    freed = sum(f.stat().st_size for f in hf_cache.rglob('*') if f.is_file()) / 1024**3
    shutil.rmtree(hf_cache, ignore_errors=True)
    print(f"✓ Cleared {hf_cache} ({freed:.1f} GB freed)")


## Done — check the footprint


In [ ]:
!du -sh "{FLUX_KLEIN_MODEL_PATH}"/*
!echo "---"
!du -sh "{FLUX_KLEIN_MODEL_PATH}"


### Getting this folder into Google Drive

Colab's `00_complete_workshop.ipynb` expects `MODELS_DIR`'s contents in a Drive folder it can share
read-only with participants (see that notebook's setup cell) — for this edition that means just the
one `FLUX.2-klein-4B-nf4` folder. A few ways to get it there, roughly fastest/most-reliable first:

1. **Google Drive desktop app**, if it's installed on this machine: drag `~/latent_vandalism_models`
   into your Drive-synced folder and let it sync in the background — simplest if it's an option.
2. **`rclone`** (if you're comfortable with it): `rclone copy ~/latent_vandalism_models remote:latent_vandalism_models`
   after `rclone config`-ing a Google Drive remote — fastest for a repeat/scripted upload.
3. **Zip + browser upload**, if neither of the above is available:
   ```bash
   cd ~ && zip -r latent_vandalism_models.zip latent_vandalism_models
   ```
   Upload the zip at [drive.google.com](https://drive.google.com), then unzip it *inside* Drive with
   a one-off cell in a Colab notebook (with Drive mounted):
   ```python
   import zipfile
   with zipfile.ZipFile('/content/drive/MyDrive/latent_vandalism_models.zip') as z:
       z.extractall('/content/drive/MyDrive/')
   ```

Once the folder is in Drive as `latent_vandalism_models`, follow the same last step as the Colab
instructor notebook: right-click it → **Share** → **"Anyone with the link"** → **Viewer** → copy the
link → paste it as `SHARED_FOLDER_LINK` in `00_complete_workshop.ipynb`.
